# Evaluación con LLM-as-judge (Gemini)

ROUGE mide solapamiento léxico, pero correlaciona mal con calidad percibida
por humanos. Usamos Gemini Flash como juez automático para evaluar las
generaciones de los mejores modelos de V3 en tres dimensiones:

- **Coherencia (1-5):** estructura lógica, sin contradicciones ni repeticiones.
- **Fidelidad (1-5):** precisión respecto al artículo fuente, sin alucinaciones.
- **Fluidez (1-5):** gramática correcta, tono periodístico natural.

Se evalúan 50 muestras por modelo sobre el test subset.

**Requisito:** variable de entorno `GEMINI_API_KEY` con una API key válida,
o pasar la key directamente en la celda de setup.

In [ ]:
# Setup
import sys
import os
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

# --- SET YOUR GEMINI API KEY HERE (or via environment variable) ---
# os.environ["GEMINI_API_KEY"] = "your-key-here"

import torch
import pandas as pd
from src.data.loader import load_config, load_cnn_dailymail
from src.evaluation.llm_judge import (
    init_gemini, judge_batch, results_to_dataframe, aggregate_scores,
)

cfg = load_config("../config/config.yaml")
dataset = load_cnn_dailymail(cfg)

FIGURES_DIR = Path("../results/figures")
TABLES_DIR = Path("../results/tables")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Generar resúmenes con los mejores modelos de V3

Cargamos el mejor checkpoint de T5 y Qwen3 identificados en V3.
**Ajustar los paths si la config ganadora fue distinta de A.**

In [ ]:
# Load best T5 checkpoint.
# ADJUST the path below to match the winning V3 config (A, B, or C).
from src.models.loader import load_model, LoadedModel
from src.evaluation.inference import generate_summaries
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

BEST_T5_DIR = "v3_t5_A"  # <-- CHANGE if a different config won

t5_ckpt = Path(f"../results/checkpoints/{BEST_T5_DIR}")
# Find checkpoint subfolder
t5_subdirs = sorted([p for p in t5_ckpt.iterdir() if p.name.startswith("checkpoint-")])
t5_path = t5_subdirs[-1] if t5_subdirs else t5_ckpt

t5_model = AutoModelForSeq2SeqLM.from_pretrained(t5_path, dtype=torch.float32)
t5_tok = AutoTokenizer.from_pretrained(t5_path, use_fast=True)
t5_model.to("cuda").eval()

t5_loaded = LoadedModel(
    model=t5_model, tokenizer=t5_tok,
    name="google/flan-t5-base", model_type="seq2seq",
    max_input_length=cfg["models"]["t5"]["max_input_length"],
)

# Generate on the first 50 test samples (enough for LLM evaluation)
N_JUDGE = 50
test_sub = dataset["test"].select(range(N_JUDGE))
articles = test_sub["article"]
references = test_sub["highlights"]

t5_preds = generate_summaries(
    t5_loaded, articles,
    max_new_tokens=cfg["dataset"]["max_target_length"],
    num_beams=cfg["generation"]["num_beams"],
    batch_size=8,
)
print(f"T5 generated {len(t5_preds)} summaries")

# Free T5 VRAM before loading Qwen
del t5_model, t5_loaded
torch.cuda.empty_cache()

In [ ]:
# Load best Qwen3 checkpoint.
# ADJUST the path below to match the winning V3 config (A, B, or C).
from peft import PeftModel

BEST_QWEN_DIR = "v3_qwen_A"  # <-- CHANGE if a different config won

qwen_base = load_model(cfg["models"]["qwen"])
qwen_ckpt = Path(f"../results/checkpoints/{BEST_QWEN_DIR}")
qwen_subdirs = sorted([p for p in qwen_ckpt.iterdir() if p.name.startswith("checkpoint-")])
qwen_path = qwen_subdirs[-1] if qwen_subdirs else qwen_ckpt

qwen_base.model = PeftModel.from_pretrained(qwen_base.model, str(qwen_path))
qwen_base.model.eval()
qwen_base.model.config.use_cache = True

qwen_preds = generate_summaries(
    qwen_base, articles,
    max_new_tokens=cfg["dataset"]["max_target_length"],
    num_beams=cfg["generation"]["num_beams"],
    batch_size=2,
)
print(f"Qwen3 generated {len(qwen_preds)} summaries")

del qwen_base
torch.cuda.empty_cache()

## 2. Evaluación con Gemini

Evaluamos cada resumen con Gemini Flash. 50 muestras × 2 modelos = 100 llamadas.
Con ~1s de delay entre llamadas, tarda ~2-3 minutos.

In [ ]:
# Initialize Gemini
gemini = init_gemini()  # reads GEMINI_API_KEY from env

# Judge T5 summaries
print("Evaluating T5 summaries...")
t5_results = judge_batch(gemini, list(articles), t5_preds, model_name="Flan-T5-base")

# Judge Qwen3 summaries
print("\nEvaluating Qwen3 summaries...")
qwen_results = judge_batch(gemini, list(articles), qwen_preds, model_name="Qwen3-1.7B")

# Combine into a single dataframe
df_judge = results_to_dataframe(t5_results + qwen_results)
df_judge.to_csv(TABLES_DIR / "llm_judge_results.csv", index=False)
print(f"\nSaved {len(df_judge)} evaluations to llm_judge_results.csv")

In [ ]:
# Aggregate scores per model
agg = aggregate_scores(df_judge)
print("\nAggregate LLM-judge scores:\n")
agg

In [ ]:
# Visualization: radar chart of mean scores by model
import matplotlib.pyplot as plt
import numpy as np

valid = df_judge[df_judge["error"] == ""].copy()
means = valid.groupby("model")[["coherence", "faithfulness", "fluency"]].mean()

categories = ["Coherence", "Faithfulness", "Fluency"]
N = len(categories)
angles = [n / N * 2 * np.pi for n in range(N)] + [0]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

for model_name, row in means.iterrows():
    values = list(row.values) + [row.values[0]]
    ax.plot(angles, values, "o-", linewidth=2, label=model_name)
    ax.fill(angles, values, alpha=0.15)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
ax.set_title("LLM-as-judge: Flan-T5-base vs Qwen3-1.7B", fontsize=13, pad=20)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "llm_judge_radar.png", bbox_inches="tight", dpi=120)
plt.show()

In [ ]:
# Show a few examples with low faithfulness (potential hallucinations)
low_faith = valid[valid["faithfulness"] <= 2].sort_values("faithfulness")
print(f"Summaries with faithfulness <= 2: {len(low_faith)}\n")

for _, row in low_faith.head(3).iterrows():
    idx = int(row["index"])
    model = row["model"]
    pred = t5_preds[idx] if "T5" in model else qwen_preds[idx]
    print(f"[{model}] idx={idx}, scores: C={row['coherence']} F={row['faithfulness']} Fl={row['fluency']}")
    print(f"Justification: {row['justification']}")
    print(f"Summary: {pred[:300]}")
    print("-" * 80)

## Análisis del LLM-as-judge

*(Rellenar tras ejecución. Puntos a cubrir:)*

1. **¿Qué modelo gana en cada dimensión?** — ROUGE no diferencia coherencia
   de fidelidad; aquí sí podemos.
2. **¿Hay alucinaciones?** — Las muestras con faithfulness bajo son candidatas
   a alucinaciones. Analizar si correlaciona con el tipo de modelo.
3. **Correlación ROUGE vs judge** — ¿los artículos con mejor ROUGE también
   reciben mejor puntuación del juez? Si no, ROUGE es insuficiente como métrica
   y este análisis aporta valor diferencial.
4. **Limitaciones** — el juez es un LLM, no un humano. Pueden haber sesgos
   sistemáticos (e.g., penalizar resúmenes cortos, favorecer cierto estilo).